In [ ]:
import sys
sys.path.append("..")

import cv2
import torch
import numpy as np
from omegaconf import OmegaConf
import matplotlib.pyplot as plt
from curvlinops import GGNLinearOperator, HessianLinearOperator, HutchinsonSquaredFrobeniusNormEstimator


from src.models.vit_classification import VisionTransformer
from src.datasets import get_dataloader

In [ ]:
config_path = "/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/results/vit_classification/mnist/config.yaml"
conf = OmegaConf.load(config_path)
train_dataloader = get_dataloader(conf.data, train=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = VisionTransformer(**conf.model.params).to(device)

# Queries

In [ ]:
params_order = [
    'transformer_blocks.0.self_attention.to_query.weight',
]

In [ ]:
param_dict = {n:p for (n, p) in model.named_parameters()}
params = [param_dict[n] for n in params_order if n in param_dict]
num_params = sum(p.numel() for p in params)
num_params_layer_all = [p.numel() for p in params]

In [ ]:
loss_function = torch.nn.CrossEntropyLoss(reduction="mean").to(device)

In [ ]:
dataloader = [next(iter(train_dataloader))]

In [ ]:
hessian_matrices = []
matrix_norms = []
hessian_norms = []

# steps = [0, 3000, 6000, 9000, 12000, 15000]
# steps = [0, 1000, 2000, 3000, 4000]
steps = [0, 200, 400, 600, 800, 1000]

for step in steps:
    if step > 0:
        model.load_state_dict(torch.load(f"/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/results/vit_classification_seed_37/mnist/checkpoints/step_{step}/model.pt", map_location="cpu", weights_only=True))
    else:
        model = VisionTransformer(**conf.model.params).to(device)
        
    param_dict = {n:p for (n, p) in model.named_parameters()}
    params = [param_dict[n] for n in params_order if n in param_dict]
    num_params = sum(p.numel() for p in params)
    num_params_layer_all = [p.numel() for p in params]
        
    hessian_linop = HessianLinearOperator(model, loss_function, params, dataloader)
    hessian_matrix = hessian_linop @ np.eye(num_params).astype(hessian_linop.dtype)
    hessian_matrices.append(hessian_matrix)
    
    matrix_norms.append(torch.block_diag(*params).norm(p=2).item())
    hessian_norms.append(np.linalg.norm(hessian_matrix, ord=2))

In [ ]:
matrices = hessian_matrices
titles = [f"Step {step}" for step in steps]
num_params_layers = [num_params_layer_all] * len(matrices)

rows, columns = 1, len(matrices)
img_width = 3

def plot(transform, transform_title=None):
    min_value = min(transform(mat).min() for mat in matrices)
    max_value = max(transform(mat).max() for mat in matrices)

    # fig, axes = plt.subplots(nrows=rows, ncols=columns, sharex=True, sharey=True)
    fig, axes = plt.subplots(nrows=rows, ncols=columns, figsize=(columns * img_width, rows * img_width))

    for idx, (ax, mat, title, num_params_layer) in enumerate(zip(axes.flat, matrices, titles, num_params_layers)):
        ax.set_title(title)
        img = ax.imshow(transform(mat), vmin=min_value, vmax=max_value)
        ax.axis("off")

        # layer blocks
        boundaries = [0] + np.cumsum(num_params_layer).tolist()
        for pos in boundaries:
            if pos not in [0, num_params]:
                style = {"color": "w", "lw": 0.5, "ls": "--", "alpha": 0.9}
                ax.axhline(y=pos - 1, xmin=0, xmax=num_params - 1, **style)
                ax.axvline(x=pos - 1, ymin=0, ymax=num_params - 1, **style)

        # colorbar
        last = idx == len(matrices) - 1
        if last:
            fig.colorbar(
                img, ax=axes.ravel().tolist(), label=transform_title, shrink=0.7
            )

    return fig, axes

In [ ]:
plt.rcParams["font.size"] = 8

In [ ]:
def logabs(mat, epsilon=1e-6):
    return np.log10(np.clip(np.abs(mat), a_min=epsilon, a_max=None))

plot(logabs, transform_title="Logarithmic absolute entries")
plt.savefig("/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/figs/hessian_entries_queries.pdf", bbox_inches="tight")
plt.show()

In [ ]:
MATRIX_NORMS = {}
MATRIX_NORMS["Queries"] = matrix_norms

HESSIAN_NORMS = {}
HESSIAN_NORMS["Queries"] = hessian_norms

# Keys

In [ ]:
params_order = [
    'transformer_blocks.0.self_attention.to_key.weight',
]

In [ ]:
param_dict = {n:p for (n, p) in model.named_parameters()}
params = [param_dict[n] for n in params_order if n in param_dict]
num_params = sum(p.numel() for p in params)
num_params_layer_all = [p.numel() for p in params]

In [ ]:
loss_function = torch.nn.CrossEntropyLoss(reduction="mean").to(device)

In [ ]:
dataloader = [next(iter(train_dataloader))]

In [ ]:
hessian_matrices = []
matrix_norms = []
hessian_norms = []

# steps = [0, 3000, 6000, 9000, 12000, 15000]
# steps = [0, 1000, 2000, 3000, 4000]
steps = [0, 200, 400, 600, 800, 1000]

for step in steps:
    if step > 0:
        model.load_state_dict(torch.load(f"/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/results/vit_classification_seed_37/mnist/checkpoints/step_{step}/model.pt", map_location="cpu", weights_only=True))
    else:
        model = VisionTransformer(**conf.model.params).to(device)
        
    param_dict = {n:p for (n, p) in model.named_parameters()}
    params = [param_dict[n] for n in params_order if n in param_dict]
    num_params = sum(p.numel() for p in params)
    num_params_layer_all = [p.numel() for p in params]
        
    hessian_linop = HessianLinearOperator(model, loss_function, params, dataloader)
    hessian_matrix = hessian_linop @ np.eye(num_params).astype(hessian_linop.dtype)
    hessian_matrices.append(hessian_matrix)
    
    matrix_norms.append(torch.block_diag(*params).norm(p=2).item())
    hessian_norms.append(np.linalg.norm(hessian_matrix, ord=2))

In [ ]:
matrices = hessian_matrices
titles = [f"Step {step}" for step in steps]
num_params_layers = [num_params_layer_all] * len(matrices)

rows, columns = 1, len(matrices)
img_width = 3

def plot(transform, transform_title=None):
    min_value = min(transform(mat).min() for mat in matrices)
    max_value = max(transform(mat).max() for mat in matrices)

    # fig, axes = plt.subplots(nrows=rows, ncols=columns, sharex=True, sharey=True)
    fig, axes = plt.subplots(nrows=rows, ncols=columns, figsize=(columns * img_width, rows * img_width))

    for idx, (ax, mat, title, num_params_layer) in enumerate(zip(axes.flat, matrices, titles, num_params_layers)):
        ax.set_title(title)
        img = ax.imshow(transform(mat), vmin=min_value, vmax=max_value)
        ax.axis("off")

        # layer blocks
        boundaries = [0] + np.cumsum(num_params_layer).tolist()
        for pos in boundaries:
            if pos not in [0, num_params]:
                style = {"color": "w", "lw": 0.5, "ls": "--", "alpha": 0.9}
                ax.axhline(y=pos - 1, xmin=0, xmax=num_params - 1, **style)
                ax.axvline(x=pos - 1, ymin=0, ymax=num_params - 1, **style)

        # colorbar
        last = idx == len(matrices) - 1
        if last:
            fig.colorbar(
                img, ax=axes.ravel().tolist(), label=transform_title, shrink=0.7
            )

    return fig, axes

In [ ]:
plt.rcParams["font.size"] = 8

In [ ]:
def logabs(mat, epsilon=1e-6):
    return np.log10(np.clip(np.abs(mat), a_min=epsilon, a_max=None))

plot(logabs, transform_title="Logarithmic absolute entries")
plt.savefig("/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/figs/hessian_entries_keys.pdf", bbox_inches="tight")
plt.show()

In [ ]:
MATRIX_NORMS["Keys"] = matrix_norms
HESSIAN_NORMS["Keys"] = hessian_norms

# Values

In [ ]:
params_order = [
    'transformer_blocks.0.self_attention.to_value.weight',
]

In [ ]:
param_dict = {n:p for (n, p) in model.named_parameters()}
params = [param_dict[n] for n in params_order if n in param_dict]
num_params = sum(p.numel() for p in params)
num_params_layer_all = [p.numel() for p in params]

In [ ]:
loss_function = torch.nn.CrossEntropyLoss(reduction="mean").to(device)

In [ ]:
dataloader = [next(iter(train_dataloader))]

In [ ]:
hessian_matrices = []
matrix_norms = []
hessian_norms = []

# steps = [0, 3000, 6000, 9000, 12000, 15000]
# steps = [0, 1000, 2000, 3000, 4000]
steps = [0, 200, 400, 600, 800, 1000]

for step in steps:
    if step > 0:
        model.load_state_dict(torch.load(f"/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/results/vit_classification_seed_37/mnist/checkpoints/step_{step}/model.pt", map_location="cpu", weights_only=True))
    else:
        model = VisionTransformer(**conf.model.params).to(device)
        
    param_dict = {n:p for (n, p) in model.named_parameters()}
    params = [param_dict[n] for n in params_order if n in param_dict]
    num_params = sum(p.numel() for p in params)
    num_params_layer_all = [p.numel() for p in params]
        
    hessian_linop = HessianLinearOperator(model, loss_function, params, dataloader)
    hessian_matrix = hessian_linop @ np.eye(num_params).astype(hessian_linop.dtype)
    hessian_matrices.append(hessian_matrix)
    
    matrix_norms.append(torch.block_diag(*params).norm(p=2).item())
    hessian_norms.append(np.linalg.norm(hessian_matrix, ord=2))

In [ ]:
matrices = hessian_matrices
titles = [f"Step {step}" for step in steps]
num_params_layers = [num_params_layer_all] * len(matrices)

rows, columns = 1, len(matrices)
img_width = 3

def plot(transform, transform_title=None):
    min_value = min(transform(mat).min() for mat in matrices)
    max_value = max(transform(mat).max() for mat in matrices)

    # fig, axes = plt.subplots(nrows=rows, ncols=columns, sharex=True, sharey=True)
    fig, axes = plt.subplots(nrows=rows, ncols=columns, figsize=(columns * img_width, rows * img_width))

    for idx, (ax, mat, title, num_params_layer) in enumerate(zip(axes.flat, matrices, titles, num_params_layers)):
        ax.set_title(title)
        img = ax.imshow(transform(mat), vmin=min_value, vmax=max_value)
        ax.axis("off")

        # layer blocks
        boundaries = [0] + np.cumsum(num_params_layer).tolist()
        for pos in boundaries:
            if pos not in [0, num_params]:
                style = {"color": "w", "lw": 0.5, "ls": "--", "alpha": 0.9}
                ax.axhline(y=pos - 1, xmin=0, xmax=num_params - 1, **style)
                ax.axvline(x=pos - 1, ymin=0, ymax=num_params - 1, **style)

        # colorbar
        last = idx == len(matrices) - 1
        if last:
            fig.colorbar(
                img, ax=axes.ravel().tolist(), label=transform_title, shrink=0.7
            )

    return fig, axes

In [ ]:
plt.rcParams["font.size"] = 8

In [ ]:
def logabs(mat, epsilon=1e-6):
    return np.log10(np.clip(np.abs(mat), a_min=epsilon, a_max=None))

plot(logabs, transform_title="Logarithmic absolute entries")
plt.savefig("/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/figs/hessian_entries_values.pdf", bbox_inches="tight")
plt.show()

In [ ]:
MATRIX_NORMS["Values"] = matrix_norms
HESSIAN_NORMS["Values"] = hessian_norms

# LayerNorm

In [ ]:
params_order = [
    'transformer_blocks.0.self_attention_norm.weight',
    'transformer_blocks.0.self_attention_norm.bias',
    'transformer_blocks.0.feed_forward_norm.weight',
    'transformer_blocks.0.feed_forward_norm.bias',
]

In [ ]:
param_dict = {n:p for (n, p) in model.named_parameters()}
params = [param_dict[n] for n in params_order if n in param_dict]
num_params = sum(p.numel() for p in params)
num_params_layer_all = [p.numel() for p in params]

In [ ]:
loss_function = torch.nn.CrossEntropyLoss(reduction="mean").to(device)

In [ ]:
dataloader = [next(iter(train_dataloader))]

In [ ]:
hessian_matrices = []
matrix_norms = []
hessian_norms = []

# steps = [0, 3000, 6000, 9000, 12000, 15000]
# steps = [0, 1000, 2000, 3000, 4000]
steps = [0, 200, 400, 600, 800, 1000]

for step in steps:
    if step > 0:
        model.load_state_dict(torch.load(f"/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/results/vit_classification_seed_37/mnist/checkpoints/step_{step}/model.pt", map_location="cpu", weights_only=True))
    else:
        model = VisionTransformer(**conf.model.params).to(device)
        
    param_dict = {n:p for (n, p) in model.named_parameters()}
    params = [param_dict[n] for n in params_order if n in param_dict]
    num_params = sum(p.numel() for p in params)
    num_params_layer_all = [p.numel() for p in params]
        
    hessian_linop = HessianLinearOperator(model, loss_function, params, dataloader)
    hessian_matrix = hessian_linop @ np.eye(num_params).astype(hessian_linop.dtype)
    hessian_matrices.append(hessian_matrix)
    
    matrix_norms.append(torch.block_diag(*params).norm(p=2).item())
    hessian_norms.append(np.linalg.norm(hessian_matrix, ord=2))

In [ ]:
matrices = hessian_matrices
titles = [f"Step {step}" for step in steps]
num_params_layers = [num_params_layer_all] * len(matrices)

rows, columns = 1, len(matrices)
img_width = 3

def plot(transform, transform_title=None):
    min_value = min(transform(mat).min() for mat in matrices)
    max_value = max(transform(mat).max() for mat in matrices)

    # fig, axes = plt.subplots(nrows=rows, ncols=columns, sharex=True, sharey=True)
    fig, axes = plt.subplots(nrows=rows, ncols=columns, figsize=(columns * img_width, rows * img_width))

    for idx, (ax, mat, title, num_params_layer) in enumerate(zip(axes.flat, matrices, titles, num_params_layers)):
        ax.set_title(title)
        img = ax.imshow(transform(mat), vmin=min_value, vmax=max_value)
        ax.axis("off")

        # layer blocks
        boundaries = [0] + np.cumsum(num_params_layer).tolist()
        for pos in boundaries:
            if pos not in [0, num_params]:
                style = {"color": "w", "lw": 0.5, "ls": "--", "alpha": 0.9}
                ax.axhline(y=pos - 1, xmin=0, xmax=num_params - 1, **style)
                ax.axvline(x=pos - 1, ymin=0, ymax=num_params - 1, **style)

        # colorbar
        last = idx == len(matrices) - 1
        if last:
            fig.colorbar(
                img, ax=axes.ravel().tolist(), label=transform_title, shrink=0.7
            )

    return fig, axes

In [ ]:
plt.rcParams["font.size"] = 8

In [ ]:
def logabs(mat, epsilon=1e-6):
    return np.log10(np.clip(np.abs(mat), a_min=epsilon, a_max=None))

plot(logabs, transform_title="Logarithmic absolute entries")
plt.savefig("/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/figs/hessian_entries_layernorm.pdf", bbox_inches="tight")
plt.show()

In [ ]:
MATRIX_NORMS["LayerNorm"] = matrix_norms
HESSIAN_NORMS["LayerNorm"] = hessian_norms

# FeedForward

In [ ]:
params_order = [
    'transformer_blocks.0.feed_forward_norm.weight',
    'transformer_blocks.0.feed_forward_norm.bias',
]

In [ ]:
param_dict = {n:p for (n, p) in model.named_parameters()}
params = [param_dict[n] for n in params_order if n in param_dict]
num_params = sum(p.numel() for p in params)
num_params_layer_all = [p.numel() for p in params]

In [ ]:
loss_function = torch.nn.CrossEntropyLoss(reduction="mean").to(device)

In [ ]:
dataloader = [next(iter(train_dataloader))]

In [ ]:
hessian_matrices = []
matrix_norms = []
hessian_norms = []

# steps = [0, 3000, 6000, 9000, 12000, 15000]
# steps = [0, 1000, 2000, 3000, 4000]
steps = [0, 200, 400, 600, 800, 1000]

for step in steps:
    if step > 0:
        model.load_state_dict(torch.load(f"/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/results/vit_classification_seed_37/mnist/checkpoints/step_{step}/model.pt", map_location="cpu", weights_only=True))
    else:
        model = VisionTransformer(**conf.model.params).to(device)
        
    param_dict = {n:p for (n, p) in model.named_parameters()}
    params = [param_dict[n] for n in params_order if n in param_dict]
    num_params = sum(p.numel() for p in params)
    num_params_layer_all = [p.numel() for p in params]
        
    hessian_linop = HessianLinearOperator(model, loss_function, params, dataloader)
    hessian_matrix = hessian_linop @ np.eye(num_params).astype(hessian_linop.dtype)
    hessian_matrices.append(hessian_matrix)
    
    matrix_norms.append(torch.block_diag(*params).norm(p=2).item())
    hessian_norms.append(np.linalg.norm(hessian_matrix, ord=2))

In [ ]:
matrices = hessian_matrices
titles = [f"Step {step}" for step in steps]
num_params_layers = [num_params_layer_all] * len(matrices)

rows, columns = 1, len(matrices)
img_width = 3

def plot(transform, transform_title=None):
    min_value = min(transform(mat).min() for mat in matrices)
    max_value = max(transform(mat).max() for mat in matrices)

    # fig, axes = plt.subplots(nrows=rows, ncols=columns, sharex=True, sharey=True)
    fig, axes = plt.subplots(nrows=rows, ncols=columns, figsize=(columns * img_width, rows * img_width))

    for idx, (ax, mat, title, num_params_layer) in enumerate(zip(axes.flat, matrices, titles, num_params_layers)):
        ax.set_title(title)
        img = ax.imshow(transform(mat), vmin=min_value, vmax=max_value)
        ax.axis("off")

        # layer blocks
        boundaries = [0] + np.cumsum(num_params_layer).tolist()
        for pos in boundaries:
            if pos not in [0, num_params]:
                style = {"color": "w", "lw": 0.5, "ls": "--", "alpha": 0.9}
                ax.axhline(y=pos - 1, xmin=0, xmax=num_params - 1, **style)
                ax.axvline(x=pos - 1, ymin=0, ymax=num_params - 1, **style)

        # colorbar
        last = idx == len(matrices) - 1
        if last:
            fig.colorbar(
                img, ax=axes.ravel().tolist(), label=transform_title, shrink=0.7
            )

    return fig, axes

In [ ]:
plt.rcParams["font.size"] = 8

In [ ]:
def logabs(mat, epsilon=1e-6):
    return np.log10(np.clip(np.abs(mat), a_min=epsilon, a_max=None))

plot(logabs, transform_title="Logarithmic absolute entries")
plt.savefig("/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/figs/hessian_entries_feedforward.pdf", bbox_inches="tight")
plt.show()

In [ ]:
MATRIX_NORMS["FeedForward"] = matrix_norms
HESSIAN_NORMS["FeedForward"] = hessian_norms

# Norms

In [ ]:
plt.rcParams["font.size"] = 14
plt.rcParams["lines.linewidth"] = 2.0

In [ ]:
plt.figure()

for layer in MATRIX_NORMS.keys():
    plt.plot(steps, MATRIX_NORMS[layer], label=layer)

plt.legend()
plt.title("Parameters norm")
plt.grid(alpha=0.1)
plt.tight_layout()
plt.savefig("/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/figs/parameters_norm.pdf", bbox_inches="tight")
plt.show()

In [ ]:
plt.figure()

for layer in HESSIAN_NORMS.keys():
    plt.plot(steps, HESSIAN_NORMS[layer], label=layer)

plt.legend()
plt.title("Hessians norm")
plt.grid(alpha=0.1)
plt.tight_layout()
plt.savefig("/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/figs/hessians_norm.pdf", bbox_inches="tight")
plt.show()